In [2]:
# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 或 ['Microsoft YaHei', 'Songti SC'] 等系统支持的字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方块的问题
# 1. 读取数据
df = pd.read_csv('F:\\机器学习\\kaohsiung_5sites_final.csv', parse_dates=['date'])
print(f"数据形状: {df.shape}")
print(f"站点列表: {df['sitename'].unique()}")

# 2. 数据清洗和预处理
print("\n=== 数据清洗 ===")

# 选择关键污染物指标
pollutants = ['pm2.5', 'pm10', 'o3', 'no2', 'so2', 'co']

# 将污染物列转换为数值类型，处理非数值数据
for col in pollutants:
    if col in df.columns:
        df[col] = df[col].astype(str)
        df[col] = df[col].replace(['', 'NA', 'N/A', 'NaN', 'nan', 'None', 'null', ' ', '  '], np.nan)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"{col}: 转换前数据类型 {df[col].dtype}, 缺失值数量 {df[col].isna().sum()}")

# 填充缺失值
for col in pollutants:
    if col in df.columns:
        df[col] = df[col].fillna(method='ffill').fillna(method='bfill').fillna(df[col].median())
        print(f"{col}: 填充后缺失值数量 {df[col].isna().sum()}")

# 3. 站点聚类（按污染特征）
print("\n=== 站点聚类分析 ===")

# 按站点计算污染物均值特征
site_features = df.groupby('sitename')[pollutants].mean()

# 填充缺失值
if site_features.isna().any().any():
    site_features = site_features.fillna(site_features.median())

# 标准化特征
site_features_scaled = StandardScaler().fit_transform(site_features)

# 使用K-means聚类
kmeans_sites = KMeans(n_clusters=3, random_state=42, n_init=10)
site_clusters = kmeans_sites.fit_predict(site_features_scaled)
site_features['cluster'] = site_clusters

# 分析每个簇的特征并分配类型
cluster_types = {}
for cluster_id in range(3):
    cluster_data = site_features[site_features['cluster'] == cluster_id]
    cluster_means = cluster_data[pollutants].mean()
    
    # 根据污染物特征判断站点类型
    if cluster_means['no2'] > cluster_means['so2']:
        cluster_types[cluster_id] = '交通主导型站点'
    elif cluster_means['so2'] > site_features[pollutants].mean()['so2']:
        cluster_types[cluster_id] = '工业影响型站点'
    else:
        cluster_types[cluster_id] = '清洁背景站点'

site_features['cluster_name'] = site_features['cluster'].map(cluster_types)

print("\n站点聚类结果:")
for site, row in site_features.iterrows():
    print(f"{site}: {row['cluster_name']}")

# 4. 时段聚类（按污染程度）
print("\n=== 时段聚类分析 ===")

# 按小时和日期聚合数据
df['hour'] = df['date'].dt.hour
df['date_only'] = df['date'].dt.date

# 创建时段特征（每小时的平均污染物浓度）
time_features = df.groupby(['date_only', 'hour'])[pollutants].mean().reset_index()

# 填充缺失值
if time_features[pollutants].isna().any().any():
    time_features[pollutants] = time_features[pollutants].fillna(time_features[pollutants].median())

# 标准化特征
time_features_scaled = StandardScaler().fit_transform(time_features[pollutants])

# 时段聚类
kmeans_times = KMeans(n_clusters=3, random_state=42, n_init=10)
time_clusters = kmeans_times.fit_predict(time_features_scaled)
time_features['cluster'] = time_clusters

# 根据PM2.5浓度定义时段类型
cluster_pm25_means = time_features.groupby('cluster')['pm2.5'].mean()
sorted_clusters = cluster_pm25_means.sort_values().index
time_period_mapping = {
    sorted_clusters[0]: '优良时段',
    sorted_clusters[1]: '轻度污染时段', 
    sorted_clusters[2]: '重污染时段'
}

time_features['time_period'] = time_features['cluster'].map(time_period_mapping)

print("\n时段聚类结果:")
print(time_features['time_period'].value_counts())

# 5. 生成站点聚类可视化图
print("\n=== 生成站点聚类可视化图 ===")

fig_site = plt.figure(figsize=(18, 12))
fig_site.suptitle('高雄空气质量站点聚类分析结果', fontsize=20, fontweight='bold', y=1.02)

# 子图1: 站点聚类（PCA降维可视化）
ax1 = plt.subplot(2, 3, 1)
pca = PCA(n_components=2)
site_pca = pca.fit_transform(site_features_scaled)

# 创建颜色映射
site_colors = {'交通主导型站点': '#FF6B6B', '工业影响型站点': '#4ECDC4', '清洁背景站点': '#45B7D1'}
for i, (site, row) in enumerate(site_features.iterrows()):
    cluster_name = row['cluster_name']
    color = site_colors[cluster_name]
    mask = site_features.index == site
    ax1.scatter(site_pca[mask, 0], site_pca[mask, 1], 
                c=[color], label=cluster_name if i==0 else "", 
                s=200, alpha=0.8, edgecolors='black', linewidth=2)
    # 添加站点标签
    ax1.annotate(site, (site_pca[mask, 0][0], site_pca[mask, 1][0]),
                xytext=(10, 10), textcoords='offset points', 
                fontsize=11, fontweight='bold', 
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax1.set_title('站点聚类可视化 (PCA降维)', fontsize=16, fontweight='bold', pad=15)
ax1.set_xlabel('主成分1 (解释方差: {:.1f}%)'.format(pca.explained_variance_ratio_[0]*100), fontsize=12)
ax1.set_ylabel('主成分2 (解释方差: {:.1f}%)'.format(pca.explained_variance_ratio_[1]*100), fontsize=12)
ax1.grid(True, alpha=0.3, linestyle='--')

# 添加图例
handles = [mpatches.Patch(color=color, label=label) for label, color in site_colors.items()]
ax1.legend(handles=handles, loc='upper left', fontsize=11, framealpha=0.9)

# 子图2: 各站点污染物浓度对比雷达图
ax2 = plt.subplot(2, 3, 2, polar=True)

angles = np.linspace(0, 2*np.pi, len(pollutants), endpoint=False).tolist()
angles += angles[:1]

for site, row in site_features.iterrows():
    values = row[pollutants].values.tolist()
    values += values[:1]
    color = site_colors[row['cluster_name']]
    ax2.plot(angles, values, marker='o', label=site, linewidth=2.5, color=color, alpha=0.7)
    ax2.fill(angles, values, alpha=0.1, color=color)

ax2.set_xticks(angles[:-1])
ax2.set_xticklabels([p.upper() for p in pollutants], fontsize=11, fontweight='bold')
ax2.set_title('各站点污染物特征雷达图', fontsize=16, fontweight='bold', pad=20)
ax2.legend(bbox_to_anchor=(1.3, 1), fontsize=10, title='站点名称', title_fontsize=11)

# 子图3: 污染物浓度热图（按站点）
ax3 = plt.subplot(2, 3, 3)
heatmap_data = site_features[pollutants].copy()
sns.heatmap(heatmap_data, annot=True, fmt='.1f', 
            cmap='YlOrRd', linewidths=0.5, ax=ax3, 
            cbar_kws={'label': '浓度值 (μg/m³)'})
ax3.set_title('各站点污染物平均浓度热图', fontsize=16, fontweight='bold', pad=15)
ax3.set_ylabel('站点名称', fontsize=12)
ax3.set_xlabel('污染物', fontsize=12)

# 子图4: 各站点污染物浓度对比条形图
ax4 = plt.subplot(2, 3, 4)
plot_data = site_features.reset_index().melt(id_vars=['sitename', 'cluster_name'], 
                                            value_vars=pollutants,
                                            var_name='污染物', value_name='浓度')

# 为每个站点分配颜色
plot_data['color'] = plot_data['cluster_name'].map(site_colors)

# 创建分组条形图
sites = plot_data['sitename'].unique()
bar_width = 0.12
x = np.arange(len(pollutants))

for i, site in enumerate(sites):
    site_data = plot_data[plot_data['sitename'] == site]
    color = site_data['color'].iloc[0]
    ax4.bar(x + i*bar_width - (len(sites)-1)*bar_width/2, 
            site_data['浓度'].values, 
            width=bar_width, label=site, color=color, alpha=0.8)

ax4.set_title('各站点污染物浓度对比', fontsize=16, fontweight='bold', pad=15)
ax4.set_xlabel('污染物', fontsize=12)
ax4.set_ylabel('平均浓度 (μg/m³)', fontsize=12)
ax4.set_xticks(x)
ax4.set_xticklabels([p.upper() for p in pollutants], fontsize=11)
ax4.legend(title='站点名称', fontsize=10, title_fontsize=11)
ax4.grid(True, alpha=0.3, axis='y')

# 子图5: 各站点类型污染物特征对比
ax5 = plt.subplot(2, 3, 5)
cluster_avg = site_features.groupby('cluster_name')[pollutants].mean()
cluster_avg.plot(kind='bar', ax=ax5, width=0.8, 
                 color=[site_colors[name] for name in cluster_avg.index])
ax5.set_title('各站点类型污染物特征对比', fontsize=16, fontweight='bold', pad=15)
ax5.set_xlabel('站点类型', fontsize=12)
ax5.set_ylabel('平均浓度 (μg/m³)', fontsize=12)
ax5.legend(title='污染物', fontsize=10, title_fontsize=11)
ax5.grid(True, alpha=0.3, axis='y')

# 子图6: 站点聚类统计信息
ax6 = plt.subplot(2, 3, 6)
ax6.axis('off')

# 添加统计信息文本
stats_text = []
stats_text.append("站点聚类统计信息")
stats_text.append("="*30)

for cluster_type in site_colors.keys():
    count = (site_features['cluster_name'] == cluster_type).sum()
    sites_list = ', '.join(site_features[site_features['cluster_name'] == cluster_type].index.tolist())
    stats_text.append(f"\n{cluster_type} ({count}个站点):")
    stats_text.append(f"  包含站点: {sites_list}")
    
    # 添加污染物特征
    cluster_data = site_features[site_features['cluster_name'] == cluster_type]
    for poll in pollutants[:3]:  # 只显示前3种主要污染物
        avg = cluster_data[poll].mean()
        stats_text.append(f"  平均{poll.upper()}: {avg:.1f} μg/m³")

stats_text.append("\n聚类分析说明:")
stats_text.append("- 基于6种污染物浓度特征")
stats_text.append("- 使用K-means聚类算法")
stats_text.append("- 标准化处理后进行聚类")

ax6.text(0.05, 0.95, '\n'.join(stats_text), 
         transform=ax6.transAxes, fontsize=11, 
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='whitesmoke', alpha=0.8))

plt.tight_layout()
plt.savefig('site_clustering_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

# 6. 生成时段聚类可视化图
print("\n=== 生成时段聚类可视化图 ===")

fig_time = plt.figure(figsize=(18, 12))
fig_time.suptitle('高雄空气质量时段聚类分析结果', fontsize=20, fontweight='bold', y=1.02)

# 创建颜色映射
time_colors = {'优良时段': '#00C851', '轻度污染时段': '#FFBB33', '重污染时段': '#FF4444'}

# 子图1: 时段聚类分布（按小时）
ax1 = plt.subplot(2, 3, 1)
hourly_pattern = time_features.groupby(['hour', 'time_period']).size().unstack()

# 确保所有时段类型都有数据
for period in time_colors.keys():
    if period not in hourly_pattern.columns:
        hourly_pattern[period] = 0

# 按正确顺序排列
hourly_pattern = hourly_pattern[list(time_colors.keys())]

hourly_pattern.plot(kind='bar', stacked=True, ax=ax1, 
                    color=[time_colors[col] for col in hourly_pattern.columns],
                    width=0.8, edgecolor='black', linewidth=0.5)

ax1.set_title('污染时段按小时分布', fontsize=16, fontweight='bold', pad=15)
ax1.set_xlabel('小时', fontsize=12)
ax1.set_ylabel('时段数量', fontsize=12)
ax1.set_xticks(range(0, 24, 2))
ax1.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 2)], rotation=45)
ax1.legend(title='时段类型', fontsize=11, title_fontsize=12)
ax1.grid(True, alpha=0.3, axis='y')

# 添加每个小时的总数
for i, hour in enumerate(hourly_pattern.index):
    total = hourly_pattern.loc[hour].sum()
    if total > 0:
        ax1.text(i, total + 0.5, str(int(total)), 
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# 子图2: 不同时段污染物浓度对比
ax2 = plt.subplot(2, 3, 2)
period_avg = time_features.groupby('time_period')[pollutants].mean()

# 确保顺序正确
period_avg = period_avg.loc[list(time_colors.keys())]

period_avg.plot(kind='bar', ax=ax2, width=0.8, 
                color=[time_colors[name] for name in period_avg.index],
                edgecolor='black', linewidth=0.5)
ax2.set_title('不同时段污染物浓度对比', fontsize=16, fontweight='bold', pad=15)
ax2.set_xlabel('时段类型', fontsize=12)
ax2.set_ylabel('平均浓度 (μg/m³)', fontsize=12)
ax2.legend(title='污染物', fontsize=10, title_fontsize=11, bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

# 子图3: 污染时段的时间分布（散点图）
ax3 = plt.subplot(2, 3, 3)
# 将日期和时间合并为datetime对象
time_features['datetime'] = pd.to_datetime(time_features['date_only'].astype(str) + ' ' + time_features['hour'].astype(str) + ':00:00')
time_features_sorted = time_features.sort_values('datetime')

# 创建颜色序列
colors_series = time_features_sorted['time_period'].map(time_colors)

scatter = ax3.scatter(time_features_sorted['datetime'], time_features_sorted['pm2.5'], 
            c=colors_series, alpha=0.7, s=80, edgecolors='black', linewidth=0.5)
ax3.set_title('污染时段时间分布 (PM2.5浓度)', fontsize=16, fontweight='bold', pad=15)
ax3.set_xlabel('日期时间', fontsize=12)
ax3.set_ylabel('PM2.5浓度 (μg/m³)', fontsize=12)
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3)

# 添加图例
legend_patches = [mpatches.Patch(color=color, label=period) 
                  for period, color in time_colors.items()]
ax3.legend(handles=legend_patches, fontsize=11, loc='upper left')

# 添加污染水平参考线
ax3.axhline(y=35, color='green', linestyle='--', alpha=0.5, label='优良阈值')
ax3.axhline(y=75, color='orange', linestyle='--', alpha=0.5, label='轻度污染阈值')
ax3.axhline(y=150, color='red', linestyle='--', alpha=0.5, label='重污染阈值')

# 子图4: 各时段PM2.5浓度分布（箱线图）
ax4 = plt.subplot(2, 3, 4)
box_data = []
labels = []
for period in time_colors.keys():
    period_data = time_features[time_features['time_period'] == period]['pm2.5']
    if len(period_data) > 0:
        box_data.append(period_data)
        labels.append(period)

bplot = ax4.boxplot(box_data, labels=labels, patch_artist=True, widths=0.6)

# 为箱线图着色
for patch, color in zip(bplot['boxes'], [time_colors[label] for label in labels]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax4.set_title('各时段PM2.5浓度分布箱线图', fontsize=16, fontweight='bold', pad=15)
ax4.set_xlabel('时段类型', fontsize=12)
ax4.set_ylabel('PM2.5浓度 (μg/m³)', fontsize=12)
ax4.grid(True, alpha=0.3, axis='y')

# 添加平均值标记
for i, period in enumerate(labels):
    mean_val = time_features[time_features['time_period'] == period]['pm2.5'].mean()
    ax4.text(i+1, mean_val, f'{mean_val:.1f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

# 子图5: 污染时段日历热图
ax5 = plt.subplot(2, 3, 5)

# 创建日期-小时网格数据
time_features['date'] = pd.to_datetime(time_features['date_only'])
pivot_data = time_features.pivot_table(index='hour', columns='date', values='pm2.5', aggfunc='mean')

# 创建热图
im = ax5.imshow(pivot_data, aspect='auto', cmap='YlOrRd', interpolation='nearest')

# 设置坐标轴
ax5.set_title('PM2.5浓度日历热图', fontsize=16, fontweight='bold', pad=15)
ax5.set_xlabel('日期', fontsize=12)
ax5.set_ylabel('小时', fontsize=12)

# 设置x轴刻度（每5天一个标签）
date_labels = pivot_data.columns.strftime('%m-%d').tolist()
ax5.set_xticks(range(0, len(date_labels), max(1, len(date_labels)//10)))
ax5.set_xticklabels([date_labels[i] for i in range(0, len(date_labels), max(1, len(date_labels)//10))], rotation=45)

# 设置y轴刻度
ax5.set_yticks(range(0, 24, 4))
ax5.set_yticklabels([f"{h:02d}:00" for h in range(0, 24, 4)])

# 添加颜色条
cbar = plt.colorbar(im, ax=ax5, pad=0.02)
cbar.set_label('PM2.5浓度 (μg/m³)', fontsize=12)

# 子图6: 时段聚类统计信息
ax6 = plt.subplot(2, 3, 6)
ax6.axis('off')

# 添加统计信息文本
stats_text = []
stats_text.append("时段聚类统计信息")
stats_text.append("="*30)

total_periods = len(time_features)
for period, color in time_colors.items():
    period_data = time_features[time_features['time_period'] == period]
    count = len(period_data)
    percentage = (count / total_periods) * 100
    
    stats_text.append(f"\n{period}:")
    stats_text.append(f"  时段数量: {count} ({percentage:.1f}%)")
    
    if count > 0:
        # 主要发生时间
        peak_hours = period_data['hour'].value_counts().head(3)
        if len(peak_hours) > 0:
            peak_hour_str = ', '.join([f"{int(h):02d}:00" for h in peak_hours.index])
            stats_text.append(f"  主要发生时间: {peak_hour_str}")
        
        # PM2.5统计
        pm25_stats = period_data['pm2.5']
        stats_text.append(f"  PM2.5范围: {pm25_stats.min():.1f}-{pm25_stats.max():.1f}")
        stats_text.append(f"  PM2.5平均: {pm25_stats.mean():.1f}")
        
        # 主要污染物
        period_means = period_data[pollutants].mean()
        main_poll = period_means.idxmax()
        stats_text.append(f"  主要污染物: {main_poll.upper()}")

stats_text.append("\n时段分类标准:")
stats_text.append("- 优良时段: PM2.5 ≤ 35 μg/m³")
stats_text.append("- 轻度污染时段: 35 < PM2.5 ≤ 75 μg/m³")
stats_text.append("- 重污染时段: PM2.5 > 75 μg/m³")

ax6.text(0.05, 0.95, '\n'.join(stats_text), 
         transform=ax6.transAxes, fontsize=11, 
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='whitesmoke', alpha=0.8))

plt.tight_layout()
plt.savefig('time_period_clustering_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

# 7. 生成结果文件
print("\n=== 保存聚类结果 ===")

# 站点聚类结果
site_results = site_features.reset_index()[['sitename', 'cluster', 'cluster_name'] + pollutants]
site_results.to_csv('site_clustering_results.csv', index=False, encoding='utf-8-sig')
print("✓ 站点聚类结果已保存: site_clustering_results.csv")

# 时段聚类结果
time_results = time_features[['date_only', 'hour', 'datetime', 'cluster', 'time_period'] + pollutants]
time_results.to_csv('time_period_clustering_results.csv', index=False, encoding='utf-8-sig')
print("✓ 时段聚类结果已保存: time_period_clustering_results.csv")

# 生成分析报告
with open('clustering_analysis_report.txt', 'w', encoding='utf-8') as f:
    f.write("高雄空气质量聚类分析报告\n")
    f.write("="*60 + "\n\n")
    
    f.write("分析时间: {}\n".format(datetime.now().strftime('%Y-%m-%d %H:%M:%S')))
    f.write("数据时间范围: {} 至 {}\n".format(df['date'].min(), df['date'].max()))
    f.write("监测站点数量: {}\n".format(len(df['sitename'].unique())))
    
    f.write("\n一、站点聚类分析结果\n")
    f.write("-"*40 + "\n")
    for site, row in site_features.iterrows():
        f.write(f"{site}: {row['cluster_name']}\n")
    
    f.write("\n二、时段聚类分析结果\n")
    f.write("-"*40 + "\n")
    for period, count in time_features['time_period'].value_counts().items():
        percentage = count/len(time_features)*100
        f.write(f"{period}: {count}个时段 ({percentage:.1f}%)\n")
    
    f.write("\n三、主要发现\n")
    f.write("-"*40 + "\n")
    
    # 站点分析
    f.write("1. 站点特征:\n")
    for cluster_type in site_colors.keys():
        sites_in_cluster = site_features[site_features['cluster_name'] == cluster_type].index.tolist()
        f.write(f"   {cluster_type}: {', '.join(sites_in_cluster)}\n")
    
    # 时段分析
    f.write("\n2. 污染时段分布:\n")
    heavy_pollution = time_features[time_features['time_period'] == '重污染时段']
    if not heavy_pollution.empty:
        peak_hours = heavy_pollution['hour'].value_counts().head(3)
        f.write(f"   重污染主要发生在: {', '.join([f'{int(h)}:00' for h in peak_hours.index])}\n")
    
    f.write("\n四、建议措施\n")
    f.write("-"*40 + "\n")
    f.write("1. 针对站点类型:\n")
    f.write("   - 交通主导型站点: 优化交通流量，推广电动汽车\n")
    f.write("   - 工业影响型站点: 加强工业排放监管\n")
    f.write("   - 清洁背景站点: 保持现有环保措施\n")
    f.write("\n2. 针对污染时段:\n")
    f.write("   - 重污染时段: 发布健康预警，减少户外活动\n")
    f.write("   - 轻度污染时段: 加强监测，采取预防措施\n")

print("\n=== 分析完成 ===")
print("已生成以下文件:")
print("1. site_clustering_visualization.png - 站点聚类可视化图")
print("2. time_period_clustering_visualization.png - 时段聚类可视化图")
print("3. site_clustering_results.csv - 站点聚类详细结果")
print("4. time_period_clustering_results.csv - 时段聚类详细结果")
print("5. clustering_analysis_report.txt - 分析报告")
# 设置字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.default'] = 'regular'  # 数学文本使用常规字体

# 然后修改代码中所有单位显示的地方：

# 1. 热图的颜色条标签
cbar_kws={'label': '浓度值 ($\mu g/m^3$)'}

# 2. Y轴标签
ax4.set_ylabel('平均浓度 ($\mu g/m^3$)', fontsize=12)
ax5.set_ylabel('平均浓度 ($\mu g/m^3$)', fontsize=12)

# 3. 散点图Y轴标签
ax3.set_ylabel('PM2.5浓度 ($\mu g/m^3$)', fontsize=12)

# 4. 统计信息中的单位
stats_text.append(f"  PM2.5范围: {pm25_stats.min():.1f}-{pm25_stats.max():.1f} $\mu g/m^3$")
stats_text.append(f"  PM2.5平均: {pm25_stats.mean():.1f} $\mu g/m^3$")

# 5. 时段统计信息
stats_text.append(f"  PM2.5范围: {pm25_stats.min():.1f}-{pm25_stats.max():.1f} $\mu g/m^3$")
stats_text.append(f"  PM2.5平均: {pm25_stats.mean():.1f} $\mu g/m^3$")

# 6. 分析报告中的单位
f.write(f"  平均{poll.upper()}: {avg:.1f} $\mu g/m^3$\n")

NameError: name 'plt' is not defined